# Sackmann <-> UK Tournament Linker

Interactive, single tour/year runner for the same workflow that backs
`scripts/mapper/build_tournament_mapping.py` - use this instead of the
terminal when you want to eyeball results before trusting them. See
[docs/scripts/tournament_linker.md](../../docs/scripts/tournament_linker.md)
for the full write-up of how matching/crosswalk/source-links work.


In [18]:
import pandas as pd

from tennis_data_pipeline.workflows.mapper import build_tournament_mapping


In [20]:
TOUR = "atp"
YEAR = 2023
REVIEW_THRESHOLD = 0.9  # matched pairs scoring below this get flagged for manual review


## Run the mapping workflow

Same call the CLI script makes - loads this tour/year's UK + Sackmann
tournament tables, matches them, and upserts
`data/mapping/tournaments/<tour>_tournament_crosswalk.csv` and
`<tour>_tournament_source_links.csv`. Existing rows in both files always win
over freshly computed ones, so rerunning this is safe even after you've
hand-corrected a row.


In [21]:
result = build_tournament_mapping(TOUR, YEAR)

print(
    f"Matched {result.matched_count}/{result.uk_total} UK and "
    f"{result.matched_count}/{result.sackmann_total} Sackmann tournaments"
)
print(f"Crosswalk:    {result.crosswalk_path}")
print(f"Source links: {result.source_links_path}")
if result.ambiguous_locations:
    print(f"Ambiguous locations (resolved via location+name): {sorted(result.ambiguous_locations)}")


Matched 65/65 UK and 65/67 Sackmann tournaments
Crosswalk:    /Users/nicholasbenelli/Workspace/repos/GitHub/-sports/-tennis/Tennis-Data-Pipeline/data/mapping/tournaments/atp_tournament_crosswalk.csv
Source links: /Users/nicholasbenelli/Workspace/repos/GitHub/-sports/-tennis/Tennis-Data-Pipeline/data/mapping/tournaments/atp_tournament_source_links.csv
Ambiguous locations (resolved via location+name): ['adelaide', 'paris']


## Review weak matches

Nothing is auto-rejected below `REVIEW_THRESHOLD` - a low score is just
worth double-checking (and hand-correcting directly in the crosswalk/
source-links CSVs if it's wrong) before relying on it downstream.


In [22]:
weak = result.review_df.loc[result.review_df["score"] < REVIEW_THRESHOLD]
print(f"{len(weak)} matched pair(s) scored below {REVIEW_THRESHOLD}")
display(weak)

print("\nAll matched pairs, weakest first:")
result.review_df


9 matched pair(s) scored below 0.9


,score,location,uk_name,sackmann_name
0,0.626667,Paris,French Open,Roland Garros
1,0.670000,Rome,Internazionali BNL d'Italia,Rome Masters
2,0.720000,Madrid,Mutua Madrid Open,Madrid Masters
3,0.724348,Miami,Miami Open,Miami Masters
4,0.772500,Turin,Masters Cup,Tour Finals
5,0.772941,Nur-Sultan,Astana Open,Astana
6,0.795000,Indian Wells,BNP Paribas Open,Indian Wells Masters
7,0.796296,Toronto,Canadian Open,Canada Masters
8,0.817143,Cincinnati,Western & Southern Financial Group Masters,Cincinnati Masters



All matched pairs, weakest first:


,score,location,uk_name,sackmann_name
0,0.626667,Paris,French Open,Roland Garros
1,0.670000,Rome,Internazionali BNL d'Italia,Rome Masters
2,0.720000,Madrid,Mutua Madrid Open,Madrid Masters
3,0.724348,Miami,Miami Open,Miami Masters
4,0.772500,Turin,Masters Cup,Tour Finals
...,...,...,...,...
60,1.000000,Halle,Halle Open,Halle
61,1.000000,Dubai,Dubai Tennis Championships,Dubai
62,1.000000,Stuttgart,Stuttgart Open,Stuttgart
63,1.000000,'s-Hertogenbosch,Rosmalen Grass Court Championships,s Hertogenbosch


## Persisted mapping tables

Read back what got written this run.


In [23]:
crosswalk = pd.read_csv(result.crosswalk_path)
print(f"{len(crosswalk)} total location keys in {result.crosswalk_path.name}")
crosswalk


75 total location keys in atp_tournament_crosswalk.csv


,location_key,atp_tournament_id,location
0,acapulco,807,Acapulco
1,adelaide,8998,Adelaide
2,adelaide|adelaide international 1,2843,Adelaide
3,adelaide|adelaide international 2,8998,Adelaide
4,almaty,9410,Almaty
...,...,...,...
70,umag,439,Umag
71,vienna,337,Vienna
72,washington,418,Washington
73,winston salem,6242,Winston-Salem


In [ ]:
source_links = pd.read_csv(result.source_links_path)
year_links = source_links.loc[source_links["year"] == YEAR]
print(
    f"{len(year_links)} source link(s) for {TOUR.upper()} {YEAR} "
    f"({len(source_links)} total across all years)"
)
year_links.sort_values(["official_tournament_id", "source"])

132 source link(s) for ATP 2023 (389 total across all years)


,atp_tournament_id,year,source,source_tournament_id
0,301,2023,sackmann,2023-0301
242,301,2023,tennis_data_uk,2023_4_auckland_asb_classic
1,308,2023,sackmann,2023-0308
216,308,2023,tennis_data_uk,2023_26_munich_bmw_open
2,311,2023,sackmann,2023-0311
...,...,...,...,...
64,9164,2023,sackmann,2023-9164
246,9164,2023,tennis_data_uk,2023_53_zhuhai_zhuhai_championships
65,9410,2023,sackmann,2023-9410
248,9410,2023,tennis_data_uk,2023_55_nur_sultan_astana_open


## Looking up a UK <-> Sackmann pair

`source_links` is long/tidy (one row per source), which is great for storage
but awkward to actually use - "what's the Sackmann id for this UK
tournament?" takes a self-join. `pivot_source_links` does that pivot: one
row per `(official_tournament_id, year)`, with a `<source>_id` column per source.

In [25]:
from tennis_data_pipeline.mapper.tournaments import pivot_source_links

wide_links = pivot_source_links(source_links)

# Look up by either side - here, a specific UK tournament's Sackmann id.
wide_links.loc[wide_links["tennis_data_uk_id"].str.contains("auckland", na=False)]


,atp_tournament_id,year,sackmann_id,tennis_data_uk_id
1,301,2023,2023-0301,2023_4_auckland_asb_classic
2,301,2024,2024-0301,2024_4_auckland_asb_classic
3,301,2025,2025-0301,2025_4_auckland_asb_classic
